# 5. 간선 X 그림자 직사각형 교차 길이 (중복 제외 처리)

## 1) import & 데이터 로드

In [1]:
from pathlib import Path
import math, pickle
import numpy as np
import pandas as pd
import networkx as nx

In [2]:
# === 셀 0: ShadowRect 정의 (언피클보다 먼저 실행!) ===
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
import math

@dataclass
class ShadowRect:
    cx: float      # 직사각형 중심 x  (center = c + (L/2)·d)
    cy: float      # 직사각형 중심 y
    length: float  # L
    width: float   # 2r
    dx: float      # d_x (태양 반대 단위벡터)
    dy: float      # d_y
    meta: Dict[str, Any]

    def endpoints(self) -> Tuple[Tuple[float,float], Tuple[float,float]]:
        """중심선 양끝점 (start, end). start는 건물 중심 c, end는 c + L·d와 거의 일치."""
        hx = 0.5 * self.length * self.dx
        hy = 0.5 * self.length * self.dy
        # 중심선 = [center - (L/2)d, center + (L/2)d]
        return (self.cx - hx, self.cy - hy), (self.cx + hx, self.cy + hy)

    def corners(self) -> List[Tuple[float, float]]:
        """
        직사각형 꼭짓점 4개를 시계방향으로 반환.
        길이 방향 단위벡터 = (dx,dy), 폭 방향 법선 = (-dy, dx).
        """
        hx = 0.5 * self.length * self.dx
        hy = 0.5 * self.length * self.dy
        nx, ny = -self.dy, self.dx
        half_w = 0.5 * self.width

        c1x, c1y = self.cx - hx, self.cy - hy   # 중심선 시작(건물 중심 쪽)
        c2x, c2y = self.cx + hx, self.cy + hy   # 중심선 끝(그림자 끝)

        p1 = (c1x - half_w*nx, c1y - half_w*ny)
        p2 = (c1x + half_w*nx, c1y + half_w*ny)
        p3 = (c2x + half_w*nx, c2y + half_w*ny)
        p4 = (c2x - half_w*nx, c2y - half_w*ny)
        return [p1, p2, p3, p4]


In [3]:
ARTIFACTS_DIR = Path("../artifacts")

with open(ARTIFACTS_DIR / "graph_xy.pkl", "rb") as f:
    G = pickle.load(f)

with open(ARTIFACTS_DIR / "shadow_rects.pkl", "rb") as f:
    shadow_rects = pickle.load(f)
    
# 태양정보가 포함된 건물 테이블 (선택적 확인용)
b_with_sun = pd.read_parquet(ARTIFACTS_DIR / "buildings_with_sun.parquet")

print("노드/간선/직사각형:", G.number_of_nodes(), G.number_of_edges(), len(shadow_rects))

노드/간선/직사각형: 330 440 87


## 2) 보조 유틸: AABB, 간선 bbox

In [4]:
from typing import Tuple, List

def edge_bbox(p0: Tuple[float,float], p1: Tuple[float,float]) -> Tuple[float,float,float,float]:
    x0,y0 = p0; x1,y1 = p1
    return (min(x0,x1), min(y0,y1), max(x0,x1), max(y0,y1))

def bbox_overlap(b1, b2) -> bool:
    # (minx,miny,maxx,maxy) 형식의 두 박스가 겹치는지
    return not (b1[2] < b2[0] or b2[2] < b1[0] or b1[3] < b2[1] or b2[3] < b1[1])


## 3) 선분 × 회전된 직사각형 교차 구간(t-interval) 구하기

아이디어: 직사각형 중심과 방향을 이용해서 로컬 좌표계로 선분을 투영하면,
직사각형은 축정렬 박스(AABB) u∈[-L/2,L/2], v∈[-W/2,W/2]가 된다.
이 프레임에서 Liang–Barsky 방식으로 t ∈ [0,1] 교차 구간을 계산한다.

In [5]:
def segment_rect_interval(p0, p1, rect) -> List[Tuple[float,float]]:
    """
    선분 p(t)=p0 + t*(p1-p0), t∈[0,1]
    rect: ShadowRect (cx,cy,length,width,dx,dy)
    반환: 교차 구간 리스트 [(t0,t1)]  (없으면 빈 리스트)
    """
    # --- 1) 로컬 좌표계 만들기 ---
    # 길이방향 단위벡터 d = (dx,dy), 폭방향 단위법선 n = (-dy,dx)
    d = np.array([rect.dx, rect.dy], dtype=float)
    n = np.array([-rect.dy, rect.dx], dtype=float)  # d에 수직

    # 중심 이동
    c = np.array([rect.cx, rect.cy], dtype=float)
    p0 = np.array(p0, dtype=float) - c
    p1 = np.array(p1, dtype=float) - c
    dp = p1 - p0  # 방향

    # --- 2) 로컬 (u,v) 좌표: u=dot(p,d), v=dot(p,n)
    u0, v0 = float(np.dot(p0, d)), float(np.dot(p0, n))
    du, dv = float(np.dot(dp, d)), float(np.dot(dp, n))

    # 직사각형 경계
    umin, umax = -0.5*rect.length, 0.5*rect.length
    vmin, vmax = -0.5*rect.width,  0.5*rect.width

    # --- 3) Liang–Barsky 스타일로 t 범위 갱신 ---
    t0, t1 = 0.0, 1.0

    # u >= umin  → du*t >= umin - u0
    p =  du; q =  umin - u0
    if p == 0 and q > 0: 
        return []
    if p != 0:
        r = q / p
        if p > 0: t0 = max(t0, r)
        else:     t1 = min(t1, r)

    # u <= umax  → du*t <= umax - u0
    p = -du; q =  umax - u0
    if p == 0 and q < 0:
        return []
    if p != 0:
        r = q / p
        if p > 0: t0 = max(t0, r)
        else:     t1 = min(t1, r)

    # v >= vmin
    p =  dv; q =  vmin - v0
    if p == 0 and q > 0:
        return []
    if p != 0:
        r = q / p
        if p > 0: t0 = max(t0, r)
        else:     t1 = min(t1, r)

    # v <= vmax
    p = -dv; q =  vmax - v0
    if p == 0 and q < 0:
        return []
    if p != 0:
        r = q / p
        if p > 0: t0 = max(t0, r)
        else:     t1 = min(t1, r)

    if t0 <= t1 and t1 >= 0 and t0 <= 1:
        return [(max(0.0, t0), min(1.0, t1))]
    return []


## 4) 구간 유니온(겹침 제거)
여러 직사각형에서 얻은 t 구간들을 정렬·병합해 중복 없는 총 길이를 만든다.

In [6]:
def merge_intervals(intervals: List[Tuple[float,float]], eps: float=1e-12):
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x[0])
    merged = [list(intervals[0])]
    for a,b in intervals[1:]:
        if a <= merged[-1][1] + eps:   # 겹치거나 닿으면 병합
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a,b])
    return [(a,b) for a,b in merged]


## 5) 간선 하나에 대한 “유일한 그늘 길이” 계산

1. AABB 프리필터로 거를 수 있는 직사각형만 후보로
2. 각 후보에 대해 segment_rect_interval로 t-구간 수집
3. merge_intervals로 유니온
4. 유니온 길이 × 간선 실제 길이(m)가 그늘 길이

In [7]:
def compute_unique_shade_length_for_edge(p0, p1, rects) -> float:
    # 간선 실길이(평면상)
    ex = p1[0] - p0[0]; ey = p1[1] - p0[1]
    edge_len = math.hypot(ex, ey)
    if edge_len == 0:
        return 0.0

    ebox = edge_bbox(p0, p1)
    intervals = []

    for r in rects:
        # 프리필터: bbox가 안 겹치면 스킵
        if hasattr(r, "corners_cache"):
            rb = r.corners_cache
        else:
            pts = r.corners()
            xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
            rb = (min(xs), min(ys), max(xs), max(ys))
            r.corners_cache = rb
        if not bbox_overlap(ebox, rb):
            continue

        # 교차 구간(t0,t1) 추출
        its = segment_rect_interval(p0, p1, r)
        intervals.extend(its)

    merged = merge_intervals(intervals)
    # t-구간 길이 합 × edge_len → 실제 미터
    shade_len = sum((b-a) for a,b in merged) * edge_len
    return float(shade_len)


## 6) 그래프 모든 간선에 적용 → shaded_len_m 속성 추가

In [8]:
from tqdm import tqdm

num_edges = G.number_of_edges()
for u, v, data in tqdm(G.edges(data=True), total=num_edges):
    p0 = (G.nodes[u]["x"], G.nodes[u]["y"])
    p1 = (G.nodes[v]["x"], G.nodes[v]["y"])
    shaded = compute_unique_shade_length_for_edge(p0, p1, shadow_rects)
    data["shaded_len_m"] = shaded

# 요약
shades = [d.get("shaded_len_m", 0.0) for _,_,d in G.edges(data=True)]
print("간선 그늘 길이 통계(m):",
      pd.Series(shades).describe().to_dict())


100%|██████████| 440/440 [00:00<00:00, 50760.34it/s]

간선 그늘 길이 통계(m): {'count': 440.0, 'mean': 1.2114693627264073, 'std': 7.179588033292065, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 66.71205492224686}


In [9]:
shades

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 2.5272766931804154,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 38.216032827614654,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 14.89769073315186,
 0.0,
 0.0,
 26.523626969175147,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 1.2701430046829079,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 39.16769132869771,
 0.0,
 42.48807857875768,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0

## 7) 저장

In [10]:
import pickle

with open(ARTIFACTS_DIR / "graph_with_shade.pkl", "wb") as f:
    pickle.dump(G, f)

print("저장:", ARTIFACTS_DIR / "graph_with_shade.pkl")


저장: ..\artifacts\graph_with_shade.pkl
